In [1]:
import pandas as pd
import numpy as np

In [ ]:
adm = pd.read_csv(r'C:admission_c.csv', parse_dates=['admission_date', 'discharge_date'])
pat = pd.read_csv(r'C:patients_c.csv', parse_dates=['registration_date'])

In [19]:
print("Admissions:", adm.shape)
print("Patients:  ", pat.shape)
print("\nAdmissions columns:", adm.columns.tolist())
print("Patients columns:  ", pat.columns.tolist())

Admissions: (13429, 10)
Patients:   (9817, 8)

Admissions columns: ['admission_id', 'patient_id', 'admission_date', 'discharge_date', 'ward_department', 'bed_id', 'discharge_status', 'readmission_flag', 'previous_admission_id', 'length_of_stay']
Patients columns:   ['patient_id', 'age', 'gender', 'blood_type', 'chronic_conditions', 'insurance_type', 'registration_date', 'treatment_adherence_pct']


In [23]:
# Extract time dimensions
adm['year'] = adm['admission_date'].dt.year
adm['month'] = adm['admission_date'].dt.month
adm['quarter'] = adm['admission_date'].dt.quarter

In [24]:
adm[['year', 'month', 'quarter']].head()

,year,month,quarter
0,2020,12,4
1,2019,6,2
2,2022,11,4
3,2017,1,1
4,2019,12,4


In [27]:
# Create bins for age groups
bins = [0, 18, 35, 50, 65, 110]
labels = ['<18', '18-35', '35-50', '50-65', '65+']

pat['age_group'] = pd.cut(pat['age'], bins=bins, labels=labels)
pat['age_group'].value_counts().sort_index()

age_group
<18      1545
18-35    1840
35-50    1616
50-65    1713
65+      2807
Name: count, dtype: int64

In [28]:
# Join admission and patient tables
df = adm.merge(pat, on='patient_id', how='left')
df.head()

,admission_id,patient_id,admission_date,discharge_date,ward_department,bed_id,discharge_status,readmission_flag,previous_admission_id,length_of_stay,...,month,quarter,age,gender,blood_type,chronic_conditions,insurance_type,registration_date,treatment_adherence_pct,age_group
0,ADM-002070,PAT-07844,2020-12-22,2021-01-02,Emergency,BED-EME-004,Recovered,False,NaN,11.0,...,12,4,46.0,Female,O-,Obesity,Medicaid,2024-03-16,73.3,35-50
1,ADM-009484,PAT-06074,2019-06-22,2019-06-28,Orthopedics,BED-ORT-026,Recovered,False,NaN,6.0,...,6,2,64.0,Male,O+,Copd,Medicare,2019-05-10,83.4,50-65
2,ADM-006331,PAT-01784,2022-11-06,NaT,Oncology,BED-ONC-027,Recovered,False,NaN,NaN,...,11,4,39.0,Male,A+,"Copd, Heart Disease",Medicaid,2021-08-02,77.6,35-50
3,ADM-001084,PAT-07425,2017-01-27,2017-02-01,Pulmonology,BED-PUL-002,Recovered,False,NaN,5.0,...,1,1,61.0,Female,O+,Copd,Private,2022-04-20,48.7,50-65
4,ADM-009785,PAT-00566,2019-12-02,2019-12-12,Neurology,BED-NEU-028,Recovered,False,NaN,10.0,...,12,4,12.0,Female,A+,Chronic Kidney Disease,Private,2022-03-16,66.7,<18


In [33]:
# Overall summary
print("Total admissions: ", len(df))
print("Unique patient nr: ", df['patient_id'].nunique())
print("AVG length of stay (days) : ", df['length_of_stay'].mean().round(1))

Total admissions:  13429
Unique patient nr:  6997
AVG length of stay (days) :  6.4


In [54]:
# Admissions by department and Year
print(df.groupby('ward_department').size().sort_values(ascending = False))
print()
print(df.groupby('year').size())

ward_department
General        2152
Emergency      1960
ICU            1502
Cardiology     1463
Pulmonology    1422
Orthopedics    1098
Pediatrics     1085
Neurology      1076
Oncology        876
Maternity       795
dtype: int64

year
2017    1170
2018    1326
2019    1495
2020    2811
2021    2449
2022    1863
2023    1314
2024    1001
dtype: int64


In [55]:
# Discharge Status:
df['discharge_status'].value_counts()

discharge_status
Recovered      12166
Transferred      689
Deceased         574
Name: count, dtype: int64

In [57]:
# Total admission # by insurance type:
df.groupby('insurance_type').size().sort_values(ascending=False)

insurance_type
Private      5654
Medicare     2890
Medicaid     2868
Uninsured    1782
dtype: int64

In [60]:
# Overall summary readmission

total_ad = len(df)
readmitted = df['readmission_flag'].sum()

print("Total admissions: ", total_ad)
print("Readmissions: ", readmitted)
print("Readmission rate: ", f"{readmitted/total_ad*100:.2f}%")

Total admissions:  13429
Readmissions:  1429
Readmission rate:  10.64%


In [65]:
# Readmission Rate by departments
df.groupby('ward_department').agg(
    total_ad = ('admission_id', 'count'),
    readmission = ('readmission_flag', 'sum')
).assign(
    rate_pct = lambda x : (x['readmission'] / x['total_ad'] * 100).round(2)
).sort_values(
    'rate_pct', ascending = False
)

,total_ad,readmission,rate_pct
ward_department,,,
Pulmonology,1422,179,12.59
ICU,1502,184,12.25
Emergency,1960,212,10.82
Pediatrics,1085,117,10.78
General,2152,224,10.41
Cardiology,1463,150,10.25
Oncology,876,89,10.16
Neurology,1076,102,9.48
Maternity,795,74,9.31


In [69]:
# Readmission Rate by Years
df.groupby('year').agg(
    total_ad = ('admission_id', 'count'),
    readmission = ('readmission_flag', 'sum')
).assign(
    rate_pct = lambda x: (x['readmission'] / x['total_ad'] * 100).round(2)
)

,total_ad,readmission,rate_pct
year,,,
2017,1170,86,7.35
2018,1326,111,8.37
2019,1495,118,7.89
2020,2811,390,13.87
2021,2449,311,12.70
2022,1863,228,12.24
2023,1314,111,8.45
2024,1001,74,7.39


In [80]:
# Patients with +2 Readmissions

high_risk_patients = df[df['readmission_flag'] == True].groupby('patient_id').agg(
    readmissions = ('admission_id', 'count'),
    departments_visited = ('ward_department', 'nunique'),
    avg_length_of_stay = ('length_of_stay', 'mean'),
    insurance = ('insurance_type', 'first'),
    age = ('age', 'first'),
    conditions = ('chronic_conditions', 'first')
).query(
    'readmissions >= 2'
).sort_values(
    'readmissions', ascending=False
)

In [82]:
print(high_risk_patients.shape)
high_risk_patients.head(10)

(73, 6)


,readmissions,departments_visited,avg_length_of_stay,insurance,age,conditions
patient_id,,,,,,
PAT-05519,4,3,3.333333,Private,41.0,Hypertension
PAT-04334,3,3,3.000000,Medicare,25.0,"Copd, Heart Disease"
PAT-06640,3,2,10.500000,Medicaid,NaN,Heart Disease
PAT-00487,3,2,3.500000,Medicare,57.0,Heart Disease
PAT-00118,2,2,5.000000,Medicare,74.0,Copd
PAT-00783,2,2,5.500000,Medicaid,32.0,Hypertension
PAT-00604,2,2,14.500000,Medicare,75.0,Asthma
PAT-00646,2,2,6.000000,Private,63.0,Copd
PAT-00711,2,2,3.500000,Uninsured,29.0,Copd
